# Integrative Industry Synthesis (Revised)
## AI-Assisted Chronic Care Triage for Diabetes Follow-Up

This notebook walks through the **revised** integrated healthcare AI workflow that combines:
1. data analysis and preprocessing on the **UCI Diabetes 130-US Hospitals (1999-2008)** dataset,
2. supervised machine learning (logistic regression + random forest, selected on held-out ROC-AUC),
3. an LLM-driven generative layer for clinical case summaries and patient outreach messages, and
4. an LLM-driven agentic routing layer that selects a follow-up tool and produces a JSON justification.

The earlier version of this artifact used synthetic data and rule-based string templates for the generative and agentic layers. Both have been replaced. See `Reflective_Synthesis_Paper.md` and the README for the full revision note.

**Before running:** copy `.env.example` to `.env` and set at least one of `OPENAI_API_KEY`, `ANTHROPIC_API_KEY`, or `LOCAL_LLM_GGUF`.

In [1]:
from pathlib import Path
import sys
import json
import pandas as pd

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
sys.path.append(str(ROOT / 'src'))

from healthcare_triage_ai import (
    load_uci_diabetes_dataset,
    preprocess_dataset,
    train_models,
    save_visuals,
    shortlist_for_llm,
    run_llm_layers,
    export_outputs,
    LLMClient,
    AGENT_TOOLS,
    LLM_SHORTLIST_SIZE,
)

## Step 1 — Load and preprocess the real dataset

The UCI dataset (~99,000 inpatient encounters across 130 U.S. hospitals) is fetched on first use and cached locally at `data/diabetic_data.csv`.

Target definition: `needs_intervention = 1` when the patient was readmitted in fewer than 30 days, and `0` otherwise.

In [2]:
raw = load_uci_diabetes_dataset()
df = preprocess_dataset(raw)
print(f'rows: {len(df):,}')
print(f'positive rate (30-day readmit): {df["needs_intervention"].mean():.3f}')
df.head()

rows: 99,340
positive rate (30-day readmit): 0.114


,patient_id,age,needs_intervention,readmitted,age_midpoint,time_in_hospital,num_lab_procedures,num_procedures,num_medications,number_outpatient,...,race,gender,A1Cresult,max_glu_serum,insulin,change,diabetesMed,admission_type_id,discharge_disposition_id,admission_source_id
0,PT-000001,[0-10),0,NO,5.0,1,41,0,1,0,...,Caucasian,Female,Missing,Missing,No,No,No,6,25,1
1,PT-000002,[10-20),0,>30,15.0,3,59,0,18,0,...,Caucasian,Female,Missing,Missing,Up,Ch,Yes,1,1,7
2,PT-000003,[20-30),0,NO,25.0,2,11,5,13,2,...,AfricanAmerican,Female,Missing,Missing,No,No,Yes,1,1,7
3,PT-000004,[30-40),0,NO,35.0,2,44,1,16,0,...,Caucasian,Male,Missing,Missing,Up,Ch,Yes,1,1,7
4,PT-000005,[40-50),0,NO,45.0,1,51,0,8,0,...,Caucasian,Male,Missing,Missing,Steady,Ch,Yes,1,1,7


## Step 2 — Train two classifiers and select the better one

Both models are evaluated on the same stratified 80/20 split, with class-weight balancing because the positive class is roughly 11% of the cohort. The model with the higher held-out ROC-AUC is used to score everyone.

In [3]:
scored_df, metrics, model = train_models(df)
save_visuals(scored_df, model, metrics)
print(json.dumps({k: v for k, v in metrics.items() if k != 'models'}, indent=2))
print('\nPer-model metrics:')
for name, info in metrics['models'].items():
    print(f'  - {name}: {info["metrics"]}')
scored_df['risk_band'].value_counts()

Training logistic regression...


Training random forest...


Selected model for scoring: random_forest (ROC-AUC=0.6711)


{
  "dataset": "UCI Diabetes 130-US Hospitals (1999-2008)",
  "records_total": 99340,
  "records_train": 79472,
  "records_test": 19868,
  "positive_rate_overall": 0.1139,
  "positive_rate_test": 0.1139,
  "selected_model": "random_forest",
  "high_or_critical_share": 0.15
}

Per-model metrics:
  - logistic_regression: {'test_accuracy': 0.6766, 'test_precision': 0.1825, 'test_recall': 0.5285, 'test_f1': 0.2713, 'test_roc_auc': 0.6655}
  - random_forest: {'test_accuracy': 0.6904, 'test_precision': 0.1914, 'test_recall': 0.5329, 'test_f1': 0.2816, 'test_roc_auc': 0.6711}


risk_band
Low         64571
Moderate    19868
High         9934
Critical     4967
Name: count, dtype: int64

## Step 3 — LLM-driven generative + agentic layers on the triage shortlist

The classifier scores all ~99k patients. The LLM layers run only on a shortlist (default 20 patients, configurable via `LLM_SHORTLIST_SIZE`). For each shortlisted patient the system produces:

- a **case summary** for the care team (LLM)
- a **patient outreach message** (LLM)
- an **agent decision** (`{action, justification, follow_up_hours}`) selected from a fixed tool set (LLM, JSON-mode)

Available agent tools:

In [4]:
for tool in AGENT_TOOLS:
    print(f"- {tool['name']}: {tool['description']}\n")

- education_reminder: Send an automated educational reminder about diabetes self-care, medication adherence, and the next routine appointment. Best for patients with stable indicators and no recent acute utilization.

- nurse_outreach: Schedule a nurse phone call within the next 5 business days to review symptoms, medication adherence, and barriers to follow-up. Best for moderate-risk patients or those with recent missed care.

- physician_review: Escalate the chart to a physician for review and possible expedited visit (within 72 hours). Best for high-risk patients or those with multiple acute utilization signals.



In [5]:
llm = LLMClient()
print(f'LLM backend: {llm.backend} ({llm.model_name})')
shortlist = shortlist_for_llm(scored_df, LLM_SHORTLIST_SIZE)
llm_df = run_llm_layers(llm, shortlist)
llm_df['agent_action'].value_counts()

LLM backend: openai (gpt-4o-mini)
Running LLM layers via backend=openai model=gpt-4o-mini on 10 patients...


  processed 5/10


  processed 10/10


agent_action
physician_review      5
nurse_outreach        4
education_reminder    1
Name: count, dtype: int64

## Step 4 — Inspect a few routed cases

In [6]:
preview = llm_df.sort_values('risk_probability', ascending=False).head(5)
for _, row in preview.iterrows():
    print('=' * 80)
    print(f"Patient {row['patient_id']} | risk={row['risk_probability']:.1%} ({row['risk_band']})")
    print(f"Agent action: {row['agent_action']} (follow up within {row['follow_up_hours']} h)")
    print(f"Justification: {row['agent_justification']}")
    print(f"Case summary:  {row['case_summary']}")
    print(f"Patient msg:   {row['patient_message']}")

Patient PT-025407 | risk=82.0% (Critical)
Agent action: physician_review (follow up within 72 h)
Justification: The patient is in a critical risk band with multiple acute utilization signals, including a high predicted readmission probability and significant glucose levels, necessitating immediate physician review.
Case summary:  The patient is a 20-30 year-old Caucasian female with a critical predicted 30-day readmission probability of 82.0%. She has a high number of inpatient visits (10) and emergency visits (4) in the last year, along with 14 medications and 9 diagnoses. Her maximum glucose serum level is over 300, and she is currently on diabetes medication with a recent medication change during this hospital stay. The team should consider closely monitoring her glucose levels and medication adherence to mitigate readmission risk.
Patient msg:   Hi there! We want to make sure you get the best care possible after your recent stay. Our care team is ready to review your situation and 

## Step 5 — Export the same artifacts as the script

In [7]:
llm_meta = {
    'backend': llm.backend,
    'model': llm.model_name,
    'shortlist_size': int(len(shortlist)),
    'tools': [tool['name'] for tool in AGENT_TOOLS],
}
export_outputs(scored_df, llm_df, metrics, llm_meta)


=== Integrated Healthcare AI Triage Demo ===
Dataset: UCI Diabetes 130-US Hospitals (1999-2008)
Records: 99,340 (train=79,472, test=19,868)
Positive rate (overall): 0.114
Selected model: random_forest
  - logistic_regression: acc=0.677, prec=0.182, rec=0.528, f1=0.271, roc_auc=0.665
  - random_forest: acc=0.690, prec=0.191, rec=0.533, f1=0.282, roc_auc=0.671

LLM backend: openai (gpt-4o-mini)
LLM-processed shortlist: 10 patients
Agent action distribution:
  - physician_review: 5
  - nurse_outreach: 4
  - education_reminder: 1

Top 3 routed cases (LLM-driven):

Patient PT-025407 | risk=82.0% (Critical) | action=physician_review (72h)
Justification: The patient is in a critical risk band with multiple acute utilization signals,
including a high predicted readmission probability and significant glucose levels, necessitating
immediate physician review.
Summary: The patient is a 20-30 year-old Caucasian female with a critical predicted 30-day
readmission probability of 82.0%. She has a hig

## Responsible AI Notes
- Real but **historical** data (1999-2008 U.S. inpatient). Generalization to current outpatient populations is not assumed.
- LLM prompts forbid diagnosis, prescription, and invented indicators.
- The agent's action space is closed and validated; invalid output is conservatively escalated to physician review.
- High-risk and ambiguous cases are routed to **human review**.
- Real deployment would require subgroup fairness analysis, governance review, an explicit PHI policy for any third-party LLM call (or use of the local `llama-cpp` backend), and ongoing monitoring.